In [2]:
import re
import math
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Dataset: 4 dokumen pendek berbahasa Indonesia (tema sistem komputer,jaringan, kecerdasan buatan, sistem temu kembali)
dokumen = [
    "Sistem temu kembali informasi sangat penting.",
    "Sistem komputer sangat penting untuk teknik.",
    "Jaringan komputer menghubungkan banyak sistem di seluruh dunia.",
    "Kecerdasan buatan mengubah cara sistem memproses informasi.",
]
nama_dokumen = [f"D{i+1}" for i in range(len(dokumen))]

# 1. Preprocessing sederhana: case folding + tokenisasi + stopwords removal
stopwords_id = {
    "yang", "dan", "di", "ke", "dari", "untuk", "pada", "dengan", "ini",
    "itu", "adalah", "akan", "tidak", "juga", "atau", "karena", "oleh",
    "dalam", "secara", "agar", "dapat", "sangat", "banyak", "seluruh",
}

def preprocess_sederhana(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [t for t in tokens if t not in stopwords_id]
    return tokens

tokens_per_dok = [preprocess_sederhana(doc) for doc in dokumen]
dokumen_bersih = [' '.join(tok) for tok in tokens_per_dok]

print("=" * 80)
print("1. HASIL PREPROCESSING SEDERHANA")
print("=" * 80)
for nama, asli, hasil in zip(nama_dokumen, dokumen, tokens_per_dok):
    print(f"{nama} | Asli   : {asli}")
    print(f"{nama} | Token  : {hasil}\n")


# 2. Bag-of-Words (raw count)
vocabulary = sorted(set(t for tokens in tokens_per_dok for t in tokens))

def hitung_bow(tokens, vocab):
    return {term: tokens.count(term) for term in vocab}

bow = [hitung_bow(tokens, vocabulary) for tokens in tokens_per_dok]
df_bow = pd.DataFrame(bow, index=nama_dokumen)[vocabulary]

print("=" * 80)
print("2. REPRESENTASI BAG-OF-WORDS (RAW COUNT)")
print("=" * 80)
print(df_bow)

# 3. Perhitungan manual: TF, DF, IDF, TF-IDF
N = len(dokumen)

# --- Term Frequency (raw count, sama seperti tabel BoW)
df_tf = df_bow.copy()

# --- Document Frequency & IDF (rumus dasar: IDF = log(N / df))
df_series = (df_bow > 0).sum(axis=0)          # df(t): jumlah dokumen yang memuat term t
idf_series = df_series.apply(lambda df_t: math.log(N / df_t))

df_idf = pd.DataFrame({
    "df(t)": df_series,
    "IDF = log(N/df)": idf_series.round(4),
})

print("\n" + "=" * 80)
print("3a. TERM FREQUENCY (TF) PER DOKUMEN")
print("=" * 80)
print(df_tf)

print("\n" + "=" * 80)
print("3b. DOCUMENT FREQUENCY (df) DAN INVERSE DOCUMENT FREQUENCY (IDF)")
print("=" * 80)
print(df_idf)

# --- TF-IDF manual = TF x IDF ---
df_tfidf_manual = df_tf.multiply(idf_series, axis=1).round(4)

print("\n" + "=" * 80)
print("3c. MATRIKS TF-IDF (MANUAL) = TF x IDF")
print("=" * 80)
print(df_tfidf_manual)

# 4. Implementasi TF-IDF menggunakan scikit-learn (TfidfVectorizer)
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(dokumen_bersih)

df_tfidf_sklearn = pd.DataFrame(
    tfidf_matrix.toarray(),
    index=nama_dokumen,
    columns=vectorizer.get_feature_names_out(),
).round(4)

print("\n" + "=" * 80)
print("4. MATRIKS TF-IDF (SCIKIT-LEARN, TfidfVectorizer)")
print("=" * 80)
print(df_tfidf_sklearn)

print("\nCatatan: scikit-learn menggunakan smooth IDF -> log(1 + N/df) + "
      "normalisasi L2 secara default, sehingga nilainya berbeda skala "
      "dengan perhitungan manual di atas, namun urutan term dengan bobot "
      "tertinggi tetap konsisten.")

# 5. Term dengan bobot TF-IDF tertinggi per dokumen (manual)
print("\n" + "=" * 80)
print("5. TERM DENGAN BOBOT TF-IDF TERTINGGI PER DOKUMEN")
print("=" * 80)

term_tertinggi = {}
for nama in nama_dokumen:
    term_top = df_tfidf_manual.loc[nama].idxmax()
    nilai_top = df_tfidf_manual.loc[nama].max()
    term_tertinggi[nama] = (term_top, nilai_top)
    print(f"{nama}: term '{term_top}' dengan bobot TF-IDF = {nilai_top:.4f}")

# 6. Analisis singkat
analisis = """
ANALISIS SINGKAT
-----------------
Term dengan bobot TF-IDF tertinggi pada tiap dokumen (D1: "kembali", D2: "teknik", D3: "dunia", D4: "buatan") adalah kata-kata yang hanya muncul
tepat satu kali dan hanya pada satu dokumen saja (df = 1), sehingga memiliki IDF paling tinggi (1.3863). Sebaliknya, kata "sistem" yang
muncul di keempat dokumen (df = 4) memperoleh IDF = 0, sehingga bobot TF-IDF-nya nol meskipun frekuensi kemunculannya (TF) paling tinggi di
setiap dokumen. Hal ini membuktikan prinsip inti TF-IDF: sebuah term dianggap penting bagi suatu dokumen bukan karena sering muncul saja,
melainkan karena ia sering muncul di dokumen itu (TF tinggi) sekaligus jarang muncul di dokumen-dokumen lain (IDF tinggi) - sehingga term
tersebutlah yang paling merepresentasikan topik unik dari masing-masing dokumen.
"""
print(analisis)

1. HASIL PREPROCESSING SEDERHANA
D1 | Asli   : Sistem temu kembali informasi sangat penting.
D1 | Token  : ['sistem', 'temu', 'kembali', 'informasi', 'penting']

D2 | Asli   : Sistem komputer sangat penting untuk teknik.
D2 | Token  : ['sistem', 'komputer', 'penting', 'teknik']

D3 | Asli   : Jaringan komputer menghubungkan banyak sistem di seluruh dunia.
D3 | Token  : ['jaringan', 'komputer', 'menghubungkan', 'sistem', 'dunia']

D4 | Asli   : Kecerdasan buatan mengubah cara sistem memproses informasi.
D4 | Token  : ['kecerdasan', 'buatan', 'mengubah', 'cara', 'sistem', 'memproses', 'informasi']

2. REPRESENTASI BAG-OF-WORDS (RAW COUNT)
    buatan  cara  dunia  informasi  jaringan  kecerdasan  kembali  komputer  \
D1       0     0      0          1         0           0        1         0   
D2       0     0      0          0         0           0        0         1   
D3       0     0      1          0         1           0        0         1   
D4       1     1      0          1     